# AURA Motor Model — Research Notebooks

These notebooks support the motor-skills modeling pipeline:

- **Dataset**: one row per session (your final `motor_sessions.csv`)
- **Goal**: train a *hostable* model that outputs an `impairment_score` in **[0,1]**
  - **0** = better motor performance in this task
  - **1** = more assistance needed in this task
- **Leakage control**: splits are grouped by **userId** (no user appears in train and test)

> ⚠️ This is a functional interaction score for this game task, **not a medical diagnosis**.


## 01 — Data QC & basic distributions

Checks:
- column presence
- missingness
- basic distributions of key motor metrics
- quick outlier views

In [ ]:
import os
import pandas as pd
import numpy as np

DATASET = os.environ.get("AURA_DATASET", "../datasets/final/motor_sessions.csv")
df = pd.read_csv(DATASET)

print("Rows:", len(df))
print("Cols:", len(df.columns))
df.head(3)


In [ ]:
required = ["sessionId","userId"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
print("✅ Required columns present:", required)


In [ ]:
na_rate = df.isna().mean().sort_values(ascending=False)
na_rate.head(30)


In [ ]:
row_na = df.isna().mean(axis=1)
print("Rows with >25% missing:", int((row_na>0.25).sum()), "of", len(df))
df.loc[row_na>0.25, ["sessionId","userId"]].head(10)


In [ ]:
key_candidates = [
    "r1_hitRate","r2_hitRate","r3_hitRate",
    "r1_reactionTime_mean","r2_reactionTime_mean","r3_reactionTime_mean",
    "r1_jerkRMS_mean","r2_jerkRMS_mean","r3_jerkRMS_mean",
    "r1_throughput_mean","r2_throughput_mean","r3_throughput_mean"
]
cols = [c for c in key_candidates if c in df.columns]
print("Found", len(cols), "key columns")
df[cols].describe().T


In [ ]:
import matplotlib.pyplot as plt

def hist(col):
    plt.figure()
    df[col].dropna().hist(bins=40)
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("count")
    plt.show()

for col in [c for c in ["r1_hitRate","r1_reactionTime_mean","r1_jerkRMS_mean","r1_throughput_mean"] if c in df.columns]:
    hist(col)


## Next

Proceed to **02_train_and_cv.ipynb** to train + cross-validate and create artifacts in `model_registry/`.
